# 06 — First-Time Calibration (No `paramstest` Required)

The five preceding notebooks all started from an existing
`paramstest.txt` that supplied seed values for L_sd, BC, tilts, and
distortion harmonics.  Real first-time-on-a-new-beamline use almost
never has that file.  This notebook shows the
[`first_time_calibrate`](../midas_calibrate_v2/pipelines/first_time.py)
pipeline that runs from **only** the calibrant material, the X-ray
wavelength, and the detector pixel/dimension specs — no operator
peak picking, no manual ring assignment, no seeded geometry.

Internally:
1. **Hough-circle BC seed** — robust to multi-panel arc
   fragmentation; votes for the (cy, cz) that the most ring pixels
   agree on.
2. **Multi-hypothesis L_sd matching** — sweep over L_sd values
   geometrically spaced around a coarse guess; for each, simulate
   ring radii and score against detected peaks.
3. **Three-stage LM refinement** — geometry → distortion →
   per-panel (optional).
4. **Reliability gates** — every attempt is scored by the
   strain-cap and basin-check gates; the pipeline returns the
   lowest-strain attempt that passes.


In [1]:
import os, time
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
from pathlib import Path
import numpy as np
from PIL import Image

from midas_calibrate_v2.pipelines.first_time import first_time_calibrate

BASE = Path(os.environ.get('V2_TEST_BASE', '/tmp/midas_v2_test'))
IMAGE = BASE / 'Ceria_63keV_900mm_100x100_0p5s_aero_0_001137.tif'
image = np.array(Image.open(str(IMAGE))).astype(np.float64)[::-1, :].copy()
print(f'image shape: {image.shape}')


image shape: (2880, 2880)


/Users/hsharma/miniconda3/envs/midas_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## What you supply — and what you don't

The cell below has only the things the user *should* know:

- The calibrant material — `lattice` + `space_group` for CeO₂
- The X-ray wavelength — from your beamline log
- The detector pixel pitch and dimensions — from manufacturer spec

You supply **no** L_sd guess (defaults to 300 mm, swept), **no** BC
guess (defaults to image center, refined by Hough), **no** tilts,
and **no** distortion seed.


In [2]:
t0 = time.time()
res = first_time_calibrate(
    image=image,
    lattice=(5.4116, 5.4116, 5.4116, 90, 90, 90),  # CeO₂ cubic
    space_group=225,                               # Fm-3m
    wavelength_A=0.196793,                         # 63 keV (typical Varex setup)
    pixel_size_um=150.0,                           # Varex 4343CT
    n_pixels_y=image.shape[0], n_pixels_z=image.shape[1],
    # No L_sd guess, no BC guess — let the auto-seed find them.
    n_iter_full=4,
    half_window_px=8.0,
    snr_min=8.0,
    trim_residual_pct=5.0,
    verbose=False,
)
print(f'\nfirst_time_calibrate elapsed: {time.time()-t0:.1f} s')


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


/Users/hsharma/opt/MIDAS/packages/midas_integrate/midas_integrate/kernels.py:396: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/Context.cpp:767.)
  return torch.sparse_csr_tensor(indptr, indices, values,
/Users/hsharma/opt/MIDAS/packages/midas_integrate/midas_integrate/kernels.py:396: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:51.)
  return torch.sparse_csr_tensor(indptr, in


first_time_calibrate elapsed: 237.2 s


## Read the result

`FirstTimeResult` carries:
- `accepted` — True if the strain-cap + basin-check gates passed
- `lsd_um`, `bc_y`, `bc_z`, `ty`, `tz` — converged geometry
- `mean_strain_uE` — final pseudo-strain
- `lsd_attempts` — list of all L_sd hypotheses tried
- `accepted_index` — which attempt won


In [3]:
unp = res.result.unpacked
print(f'L_sd:            {float(unp["Lsd"]):.2f} µm  ({float(unp["Lsd"])/1000:.2f} mm)')
print(f'BC:              ({float(unp["BC_y"]):.3f}, {float(unp["BC_z"]):.3f}) px')
print(f'tilts:           ty = {float(unp["ty"]):+.4f}°,  tz = {float(unp["tz"]):+.4f}°')
print(f'final strain:    {res.result.history[-1].mean_strain_uE:.2f} µε')
print(f'L_sd attempts:   {len(res.lsd_attempts)} tried '
      f'({[round(v) for v in res.lsd_attempts]} µm)')
print(f'\ndiagnostics:')
for d in res.diagnostics.results:
    print(f'  [{d.severity:<8s}] {d.name}: {d.message}')
print(f'\nstage log:')
for line in res.stage_log:
    print(f'  {line}')


L_sd:            895928.60 µm  (895.93 mm)
BC:              (1446.977, 1468.912) px
tilts:           ty = -0.3107°,  tz = +0.3853°
final strain:    8.62 µε
L_sd attempts:   1 tried ([895872] µm)

diagnostics:
  [ok      ] strain_cap: converged strain 8.62 μϵ — within calibrant range
  [ok      ] basin_check: seed-to-MAP drift: ΔLsd=+57 μm (+0.01 %), ΔBC=1.7 px — within safe basin
  [fail    ] cross_validation: held-out median 12.17 μϵ vs train 3.52 μϵ (+3.46× , KS p=5.12e-47) — analytical basis is incomplete on rings ≥10

stage log:
  stage1 Lsd→895784 μm  strain=1092.4 μϵ
  stage2a BC=(1447.5,1468.4) strain=527.5 μϵ
  stage2b strain=59.2 μϵ BC=(1447.0,1468.9)  tilts=(-0.291,+0.409)
  stage3 strain=8.62 μϵ Lsd=895928.6
  stage1 Lsd→895784 μm  strain=1092.4 μϵ
  stage2a BC=(1447.5,1468.4) strain=527.5 μϵ
  stage2b strain=59.2 μϵ BC=(1447.0,1468.9)  tilts=(-0.291,+0.409)
  stage3 strain=8.62 μϵ Lsd=895928.6


## Compare to the paramstest-seeded version

The paramstest-seeded run in notebook 01 converged on the same
image to L_sd ≈ 895925 µm, BC ≈ (1447, 1469) with strain 7.74 µε.
The first-time auto-seed should land within ~50 µm in L_sd and
~1 px in BC of those values without any prior knowledge.

If the auto-seed gives a much worse strain (>100 µε), it failed —
typically because the chosen L_sd hypothesis sweep didn't bracket
the true value (try a broader `lsd_sweep_factors` argument), or
because the calibrant has very few intense rings.  The
`gate_verdict` field tells you *why* an attempt failed.

## When to use this vs `autocalibrate_pv`

| Use first-time-calibrate when | Use autocalibrate_pv when |
|---|---|
| New beamline / first session of a new detector | You have a recent paramstest from the same setup |
| You don't trust the seed in your paramstest | You're refining a known geometry across many images |
| You want the auto-seed + sweep to bracket L_sd | You know L_sd to ~1 % already |

For a production stable beamline, `autocalibrate_pv` from a fresh
paramstest is faster (~30 s vs ~2 min for the auto-seed sweep).
